In [0]:
import requests
import json
import time
from datetime import datetime, timezone

# Config from ../04_utils/shared_config (inlined because %run path resolution fails on serverless)
CATALOG = "workspace"
SCHEMA = "crypto_live"
VOLUME_NAME = "raw_landing"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

COIN_IDS = [
    "bitcoin", "ethereum", "tether", "binancecoin", "solana",
    "ripple", "cardano", "dogecoin", "polkadot", "litecoin"
]

def fetch_prices_with_retry(coin_ids, max_retries=3, backoff_seconds=5):
    """
    Calls the CoinGecko API for live prices, with retry logic for real network failures.
    This mirrors the 'Error Handling & Retry' pattern from your pipeline notes —
    except this time, failures are genuinely possible (rate limits, timeouts),
    not simulated.
    """
    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {
        "ids": ",".join(coin_ids),
        "vs_currencies": "usd",
        "include_market_cap": "true",
        "include_24hr_vol": "true",
        "include_24hr_change": "true",
        "include_last_updated_at": "true"
    }

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()   # raises an exception for 4xx/5xx responses
            print(f"API call succeeded on attempt {attempt}")
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                print(f"Retrying in {backoff_seconds} seconds...")
                time.sleep(backoff_seconds)
            else:
                print("All retry attempts exhausted. Raising the error.")
                raise

# Fetch the live data
raw_data = fetch_prices_with_retry(COIN_IDS)

# Add a snapshot timestamp so we know exactly when this batch was captured
snapshot_time = datetime.now(timezone.utc).isoformat()
raw_data["_snapshot_timestamp"] = snapshot_time

# Land it as a timestamped JSON file in the Volume — this is what Auto Loader will pick up next
file_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
file_path = f"{VOLUME_PATH}/prices_{file_timestamp}.json"

with open(file_path, "w") as f:
    json.dump(raw_data, f)

print(f"Landed snapshot: {file_path}")
print(f"Coins captured: {len(COIN_IDS)}")

API call succeeded on attempt 1
Landed snapshot: /Volumes/workspace/crypto_live/raw_landing/prices_20260920_161806.json
Coins captured: 10
